# C07 · Races, memory ordering and a lock-free queue

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/01_c_bare_metal/C07_races_memory_ordering_and_a_lock_free_queue.ipynb)

**Track:** C and bare-metal · **Skill axis (sheet):** C and bare-metal · **Time:** about 60 min

**Skill Hub concepts:** Race condition and atomicity; Memory barrier and compiler reordering; Lock-free single-producer single-consumer queue

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("C07")

## Map: why `count++` loses updates, and what each fix guarantees

```
 Thread A            Thread B           count
 load  r = 5                              5
                     load  r = 5          5
 store 6                                  6
                     store 6              6    <- one increment lost
```

| Tool | Atomic read-modify-write | Stops compiler reordering | Stops CPU reordering | Cost |
|---|---|---|---|---|
| `volatile` | no | only among volatile accesses | no | none |
| `atomic_signal_fence` / `asm("" ::: "memory")` | no | yes | no | none |
| `memory_order_relaxed` atomic | yes | for that variable only | no | low |
| release store / acquire load pair | yes | yes, across the pair | yes, across the pair | low |
| `memory_order_seq_cst` (default) | yes | yes | yes, one global order | highest |
| disable interrupts (single core) | yes, for the section | with a memory clobber | n/a on one core | latency |

Single-producer single-consumer ring: the producer is the only writer of `head`, the consumer the only writer of
`tail`. The producer writes the slot, **then** publishes `head` with a release store; the consumer reads `head` with
an acquire load, **then** reads the slot. Free-running indices: count = `head - tail`, slot = `index & (N - 1)`.

## Task 1: see the race (given), then measure it

In [ ]:
%%writefile race.c
#include <pthread.h>
#include <stdatomic.h>
#include <stdio.h>
#define N 1000000
int plain; atomic_int atom;
void *work(void *a) { for (int i = 0; i < N; i++) { plain++; atomic_fetch_add_explicit(&atom, 1, memory_order_relaxed); } return a; }
int main(void) {
    pthread_t t[2];
    for (int i = 0; i < 2; i++) pthread_create(&t[i], 0, work, 0);
    for (int i = 0; i < 2; i++) pthread_join(t[i], 0);
    printf("plain=%d atomic=%d expected=%d\n", plain, atomic_load(&atom), 2 * N);
}

In [ ]:
from roboprep import clab
for _ in range(3):
    print(clab.compile_run("race.c", flags="-O1"))

## Task 2: the lock-free single-producer single-consumer queue
`spsc.h` is given. The checker runs single-thread tests, then a producer and a consumer thread passing one
million numbers under ThreadSanitizer, which reports any unsynchronized access.

In [ ]:
%%writefile spsc.h
#include <stdatomic.h>
#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>
#define QCAP 64u   /* power of two */
typedef struct {
    uint32_t buf[QCAP];
    _Atomic size_t head;   /* written only by the producer */
    _Atomic size_t tail;   /* written only by the consumer */
} spsc_t;
bool spsc_push(spsc_t *q, uint32_t v);
bool spsc_pop(spsc_t *q, uint32_t *out);

In [ ]:
%%writefile spsc.c
#include "spsc.h"

bool spsc_push(spsc_t *q, uint32_t v) {
    (void)q; (void)v; return false; /* TODO */
}

bool spsc_pop(spsc_t *q, uint32_t *out) {
    (void)q; (void)out; return false; /* TODO */
}

In [ ]:
lab.check("spsc")

## Checkpoint: Math
**M1 (Apply).** A ring with 16 slots that uses the "keep one slot empty" rule (index-based, not free-running).
Maximum number of stored items?

**M2 (Analyze).** Two threads each run `count++` (non-atomic load, add, store) 1,000,000 times, starting from 0.
Smallest final value that some interleaving can produce?

**M3 (Apply).** Bytes of buffer for a 64-entry ring of 12-byte messages?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Analyze).** Store-buffer test: `x = y = 0`; thread 1 does `x = 1; r1 = y;`, thread 2 does `y = 1; r2 = x;`.
With `memory_order_relaxed` for all four accesses, can the run end with `r1 == 0 && r2 == 0`?
- a) never
- b) yes; only `seq_cst` (or a full fence between the store and the load) rules it out
- c) only on Arm, never on x86

**K2 (Evaluate).** Explain why `volatile` is not enough for the SPSC queue across two cores, and what the release and
acquire pair guarantees that `volatile` does not.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `c07.h`).
3. In `roboprep/checks/c07.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()